# Movie Recommendation System Notebook

<a href="https://colab.research.google.com/github/Shivambharti28/Movie-Recommendation-System/blob/main/Movies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook documents the content-based recommendation pipeline used by the project. It shows where the data comes from, why TF-IDF was chosen, which features are merged into the model, and what the similarity scores look like for a real example.

## Dataset Source

The local model is rebuilt from [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) by Rounak Banik.

This project uses three raw files from that dataset:

- `movies_metadata.csv`
- `credits.csv`
- `keywords.csv`

Place those files in `data/raw/` before running the notebook or `build_artifacts.py`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from build_artifacts import build_feature_frame, configure_nltk, read_source_data

In [ ]:
movies, credits, keywords = read_source_data()

pd.DataFrame(
    {
        'file': ['movies_metadata.csv', 'credits.csv', 'keywords.csv'],
        'rows': [len(movies), len(credits), len(keywords)],
        'columns': [movies.shape[1], credits.shape[1], keywords.shape[1]],
    }
)

## Why TF-IDF?

TF-IDF is a good fit here because this is a content-based recommender built from movie metadata rather than user ratings. It rewards terms that are informative for a movie while down-weighting generic words that appear everywhere.

In this project, TF-IDF works well because it can compare movies using descriptive text such as genres, cast names, directors, keywords, and plot summaries without requiring a separate collaborative-filtering dataset.

## Features Used In The Model

The old pipeline leaned too heavily on `overview`. The updated pipeline builds a weighted `tags` field from multiple metadata sources:

- `overview` for plot context
- `genres` for broad thematic similarity
- `cast` for recurring actor-based matches
- `director` for strong stylistic signals
- `keywords` for specific motifs such as dreams, heists, or space travel

The text is then cleaned with lowercasing, punctuation removal, English stop-word removal, and lemmatization before TF-IDF is fitted.

In [ ]:
stop_words, lemmatizer = configure_nltk()
feature_df = build_feature_frame(
    movies=movies,
    credits=credits,
    keywords=keywords,
    stop_words=stop_words,
    lemmatizer=lemmatizer,
)

feature_df[['title', 'genres', 'cast', 'director', 'keywords', 'tags']].head()

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True,
)

tfidf_matrix = tfidf.fit_transform(feature_df['tags'])
indices = pd.Series(feature_df.index, index=feature_df['title']).drop_duplicates()

tfidf_matrix.shape

## What The Similarity Scores Mean

Cosine similarity compares the TF-IDF vector for one movie against every other movie in the dataset. A higher score means the movies share more weighted terms in the feature space.

The exact score is less important than the ranking. For example, if `Interstellar` and `The Dark Knight` both land in `Inception`'s top 10, the ranking is behaving much more like a human would expect from a Christopher Nolan-centered content model.

In [ ]:
def recommend_with_scores(title: str, n: int = 10) -> pd.DataFrame:
    if title not in indices:
        raise KeyError(f'{title!r} not found in the dataset')

    idx = int(indices[title])
    scores = (tfidf_matrix @ tfidf_matrix[idx].T).toarray().ravel()
    order = scores.argsort()[::-1]

    rows = []
    for candidate_idx in order:
        if int(candidate_idx) == idx:
            continue
        rows.append(
            {
                'title': feature_df.iloc[int(candidate_idx)]['title'],
                'score': float(scores[int(candidate_idx)]),
            }
        )
        if len(rows) >= n:
            break

    return pd.DataFrame(rows)


inception_recommendations = recommend_with_scores('Inception', 10)
inception_recommendations

## Sanity Check

A quick qualitative test is useful even in a small content-based project. For `Inception`, two strong Nolan neighbors should appear in the top 10 recommendations:

- `The Dark Knight`
- `Interstellar`

This check is intentionally simple, but it catches obvious regressions if the feature engineering or preprocessing changes in a bad direction.

In [ ]:
recommendation_titles = inception_recommendations['title'].tolist()

assert 'The Dark Knight' in recommendation_titles
assert 'Interstellar' in recommendation_titles

recommendation_titles

## Rebuilding The Project Artifacts

The application loads prebuilt pickle files, so the final step is to save the DataFrame, title index, fitted TF-IDF vectorizer, and sparse TF-IDF matrix.

The project exposes the same pipeline in `build_artifacts.py`, which is the reproducible source of truth for these files.

In [ ]:
from build_artifacts import build_and_save_artifacts

build_and_save_artifacts()